# Sub‑step 5.1: Core Feature Derivation

In [1]:
print("\nSTEP 5 – SUB‑STEP 5.1: CORE FEATURE DERIVATION\n")

import pandas as pd
import numpy as np

# ----------------------------------
# Load cleaned dataset from Step 4
# ----------------------------------
fact = pd.read_parquet("maplecare_fact_claim_line_cleaned.parquet")

print(f"📦 Input rows: {fact.shape[0]}")
print(f"📦 Input columns: {fact.shape[1]}")

# ----------------------------------
# A. Financial Ratio Features
# ----------------------------------
fact["allowed_to_charge_ratio"] = (
    fact["allowed_amount"] / fact["charge_amount"]
).replace([np.inf, -np.inf], np.nan)

fact["paid_to_allowed_ratio"] = (
    fact["paid_amount"] / fact["allowed_amount"]
).replace([np.inf, -np.inf], np.nan)

fact["var_to_charge_ratio"] = (
    fact["value_at_risk"] / fact["charge_amount"]
).replace([np.inf, -np.inf], np.nan)

print("✅ Financial ratio features created.")

# ----------------------------------
# B. Core Denial & Risk Flags
# ----------------------------------
fact["is_denied_flag"] = fact["status"] == "Denied"

# High‑value claim flag (top 20% by charge)
high_value_threshold = fact["charge_amount"].quantile(0.80)
fact["high_value_claim_flag"] = fact["charge_amount"] >= high_value_threshold

# High value‑at‑risk flag (top 20%)
high_var_threshold = fact["value_at_risk"].quantile(0.80)
fact["high_var_flag"] = fact["value_at_risk"] >= high_var_threshold

print("✅ Denial and risk flags created.")

# ----------------------------------
# C. Submission Timing Buckets
# ----------------------------------
def lag_bucket(days):
    if days <= 7:
        return "0–7 days"
    elif days <= 15:
        return "8–15 days"
    elif days <= 30:
        return "16–30 days"
    else:
        return "30+ days"

fact["submission_lag_bucket"] = fact["submit_lag_days"].apply(lag_bucket)

print("✅ Submission lag buckets created.")

# ----------------------------------
# Quick sanity check
# ----------------------------------
print("\n🔍 Sample engineered features:")
display_cols = [
    "charge_amount",
    "allowed_to_charge_ratio",
    "paid_to_allowed_ratio",
    "var_to_charge_ratio",
    "is_denied_flag",
    "high_value_claim_flag",
    "high_var_flag",
    "submission_lag_bucket"
]
print(fact[display_cols].head())

print("\n✅ Step 5.1 Core Feature Derivation completed.")


STEP 5 – SUB‑STEP 5.1: CORE FEATURE DERIVATION

📦 Input rows: 120000
📦 Input columns: 19
✅ Financial ratio features created.
✅ Denial and risk flags created.
✅ Submission lag buckets created.

🔍 Sample engineered features:
   charge_amount  allowed_to_charge_ratio  paid_to_allowed_ratio  \
0     202.827257                 0.702589               0.826615   
1   13284.159346                 0.731211               0.785102   
2     181.238547                 0.682935               0.822374   
3     183.065056                 0.583518               0.851238   
4   15795.533530                 0.674471               0.905662   

   var_to_charge_ratio  is_denied_flag  high_value_claim_flag  high_var_flag  \
0             0.419229            True                  False          False   
1             0.425925            True                  False           True   
2             0.438373           False                  False          False   
3             0.503287           False         

# Sub‑step 5.2: Entity‑Level & Historical Feature Engineering

In [2]:
print("\nSTEP 5 – SUB‑STEP 5.2: ENTITY‑LEVEL & HISTORICAL FEATURE ENGINEERING\n")

import pandas as pd
import numpy as np

# ----------------------------------
# Load data from Step 5.1 output
# ----------------------------------
fact = fact.copy()  # assumes Step 5.1 just ran in the same notebook/session

# ----------------------------------
# A. Payer‑Level Aggregates
# ----------------------------------
payer_agg = (
    fact
    .groupby("payer_sk")
    .agg(
        payer_claim_count=("claim_line_id_nat", "count"),
        payer_denial_rate=("is_denied_flag", "mean"),
        payer_avg_charge=("charge_amount", "mean"),
        payer_avg_var=("value_at_risk", "mean")
    )
    .reset_index()
)

fact = fact.merge(payer_agg, on="payer_sk", how="left")

print("✅ Payer‑level features created.")

# ----------------------------------
# B. Facility‑Level Aggregates
# ----------------------------------
facility_agg = (
    fact
    .groupby("facility_sk")
    .agg(
        facility_claim_count=("claim_line_id_nat", "count"),
        facility_denial_rate=("is_denied_flag", "mean"),
        facility_avg_charge=("charge_amount", "mean")
    )
    .reset_index()
)

fact = fact.merge(facility_agg, on="facility_sk", how="left")

print("✅ Facility‑level features created.")

# ----------------------------------
# C. Provider‑Level Aggregates
# ----------------------------------
provider_agg = (
    fact
    .groupby("provider_sk")
    .agg(
        provider_claim_count=("claim_line_id_nat", "count"),
        provider_denial_rate=("is_denied_flag", "mean")
    )
    .reset_index()
)

fact = fact.merge(provider_agg, on="provider_sk", how="left")

print("✅ Provider‑level features created.")

# ----------------------------------
# D. CPT‑Level Risk Behavior
# ----------------------------------
cpt_agg = (
    fact
    .groupby("cpt_sk")
    .agg(
        cpt_denial_rate=("is_denied_flag", "mean"),
        cpt_avg_charge=("charge_amount", "mean")
    )
    .reset_index()
)

fact = fact.merge(cpt_agg, on="cpt_sk", how="left")

print("✅ CPT‑level features created.")

# ----------------------------------
# E. ICD‑Level Risk Behavior
# ----------------------------------
icd_agg = (
    fact
    .groupby("icd_sk")
    .agg(
        icd_denial_rate=("is_denied_flag", "mean")
    )
    .reset_index()
)

fact = fact.merge(icd_agg, on="icd_sk", how="left")

print("✅ ICD‑level features created.")

# ----------------------------------
# Quick validation
# ----------------------------------
print("\n🔍 Sample entity‑level features:")
sample_cols = [
    "payer_denial_rate",
    "facility_denial_rate",
    "provider_denial_rate",
    "cpt_denial_rate",
    "icd_denial_rate"
]
print(fact[sample_cols].head())




STEP 5 – SUB‑STEP 5.2: ENTITY‑LEVEL & HISTORICAL FEATURE ENGINEERING

✅ Payer‑level features created.
✅ Facility‑level features created.
✅ Provider‑level features created.
✅ CPT‑level features created.
✅ ICD‑level features created.

🔍 Sample entity‑level features:
   payer_denial_rate  facility_denial_rate  provider_denial_rate  \
0           0.412116              0.434705              0.447176   
1           0.412116              0.434705              0.426284   
2           0.412116              0.431351              0.428571   
3           0.427955              0.436729              0.447176   
4           0.427955              0.431351              0.412942   

   cpt_denial_rate  icd_denial_rate  
0         0.326942         0.475353  
1         0.552309         0.475353  
2         0.356866         0.369030  
3         0.326942         0.450310  
4         0.575398         0.369030  


# Sub‑step 5.3: Interaction & Risk Concentration Features

In [5]:
# ----------------------------------
# B. Risk Tier Buckets (Corrected)
# ----------------------------------

# Value at risk tiers (unchanged – continuous enough)
fact["var_risk_tier"] = pd.qcut(
    fact["value_at_risk"],
    q=[0, 0.6, 0.85, 1.0],
    labels=["Low", "Medium", "High"]
)

# ✅ Payer denial risk tiers using rank (avoids qcut duplicates issue)
payer_rank = fact["payer_denial_rate"].rank(method="average", pct=True)

fact["payer_risk_tier"] = pd.cut(
    payer_rank,
    bins=[0, 0.6, 0.85, 1.0],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

print("✅ Risk tier buckets created (rank‑based for stability).")

✅ Risk tier buckets created (rank‑based for stability).


# Sub‑step 5.4: Final Analytics‑Ready Feature Table

In [7]:
print("\nSTEP 5 – SUB‑STEP 5.4: FINAL ANALYTICS‑READY FEATURE TABLE\n")

import pandas as pd

# ----------------------------------
# Select required columns
# ----------------------------------
final_feature_columns = [
    # Identifiers
    "claim_line_id_nat",
    "service_date_sk",
    "patient_sk",
    "provider_sk",
    "facility_sk",
    "payer_sk",
    "cpt_sk",
    "icd_sk",

    # Outcome / reference
    "status",
    "is_denied_flag",

    # Raw financials
    "charge_amount",
    "allowed_amount",
    "paid_amount",
    "denial_amount",
    "value_at_risk",

    # Core engineered ratios (Step 5.1)
    "allowed_to_charge_ratio",
    "paid_to_allowed_ratio",
    "var_to_charge_ratio",

    # Risk flags (Step 5.1 & 5.3)
    "high_value_claim_flag",
    "high_var_flag",
    "high_value_late_flag",
    "high_payer_risk_flag",
    "procedure_coding_risk_flag",

    # Timing & operational
    "submit_lag_days",
    "submission_lag_bucket",
    "has_prior_authorization",
    "coding_completeness_score",

    # Entity‑level behavior (Step 5.2)
    "payer_denial_rate",
    "facility_denial_rate",
    "provider_denial_rate",
    "cpt_denial_rate",
    "icd_denial_rate",

    # Risk tiers (Step 5.3)
    "var_risk_tier",
    "payer_risk_tier",
]

final_features = fact[final_feature_columns].copy()

# ----------------------------------
# Final validation
# ----------------------------------
print(f"✅ Final feature table rows: {final_features.shape[0]}")
print(f"✅ Final feature table columns: {final_features.shape[1]}")

# ----------------------------------
# Export for ML & BI
# ----------------------------------
output_file = "maplecare_claims_feature_table.parquet"
final_features.to_parquet(output_file, index=False)

print(f"\n✅ Analytics‑ready feature table exported: {output_file}")
print("✅ Step 5.4 completed successfully.")


STEP 5 – SUB‑STEP 5.4: FINAL ANALYTICS‑READY FEATURE TABLE

✅ Final feature table rows: 120000
✅ Final feature table columns: 34

✅ Analytics‑ready feature table exported: maplecare_claims_feature_table.parquet
✅ Step 5.4 completed successfully.
